In [150]:
import os
import json
import pandas as pd
import numpy as np
import geopandas as gpd
import subprocess
from tqdm import tqdm

In [151]:
R2D_E1_tempdir = "/resnick/groups/enceladus/jyzhao/cybershake_ngmm_prediction/risk_calculation/Exploring_code/tmp.SimCenter_E1"
R2D_E1_config_fp = os.path.join(R2D_E1_tempdir, "inputRWHALE.json")
with open(R2D_E1_config_fp, 'r') as f:
    R2D_E1_config = json.load(f)
pypsa_config = R2D_E1_config.copy()
num_samples = 1000

In [152]:
asset_bau_fp = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/eq_handoff_fullcats/bau/disagg/assets_all.csv"
asset_bau_df = pd.read_csv(asset_bau_fp)
bau_substations = asset_bau_df[asset_bau_df['asset_type'] != 'storage_unit']
bau_substations = bau_substations[bau_substations['new_mw'] == 0]
bau_substations_loc = bau_substations[['lon', 'lat']].apply(
    lambda x: (x['lon'], x['lat']),
    axis=1
)

In [ ]:
eq_scenario_id = "bssc2014haywardhnhsshaw09mod_m7p32_se"
expansion_scenario = "aggressive"

In [154]:
eq_scenario_dir = f"/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/{eq_scenario_id}"
asset_gm_fp = os.path.join(eq_scenario_dir, expansion_scenario, "assets_all_with_pga.csv")
asset_gm_df = pd.read_csv(asset_gm_fp)
asset_gm_df.columns


Index(['scenario', 'network_kind', 'plant_id', 'asset_type', 'bus', 'lon',
       'lat', 'v_nom_kv', 'region', 'bus_resolution', 'carrier',
       'damage_category', 'p_nom_before_mw', 'p_nom_after_mw', 'new_mw',
       'is_new_build', 'p_nom_min', 'p_nom', 'p_nom_extendable',
       'ramp_limit_up', 'ramp_limit_down', 'efficiency', 'marginal_cost',
       'capital_cost', 'build_year', 'lifetime', 'control', 'p_nom_max',
       'p_max_pu', 'weight', 'vom_cost', 'fuel_cost', 'heat_rate',
       'carrier_base', 'land_region', 'committable', 'built_mw', 'type',
       'p_nom_mod', 'p_min_pu', 'p_set', 'q_set', 'sign',
       'marginal_cost_quadratic', 'start_up_cost', 'shut_down_cost',
       'stand_by_cost', 'min_up_time', 'min_down_time', 'up_time_before',
       'down_time_before', 'ramp_limit_start_up', 'ramp_limit_shut_down',
       'p_nom_opt', 'color_key', 'max_hours', 'efficiency_store',
       'efficiency_dispatch', 'cyclic_state_of_charge',
       'state_of_charge_initial_per_pe

In [155]:
asset_gm_df['asset_type'].value_counts()

asset_type
generator       3020
storage_unit     270
Name: count, dtype: int64

In [156]:
# Create pelicun readable inventory csv files
asset_gm_df['id'] = asset_gm_df['asset_id'].astype(str)
# Pelicun assume the first column is 'id'
id_col = asset_gm_df.pop('id')
asset_gm_df.insert(0, 'id', id_col)

substation_mask = asset_gm_df[['lon', 'lat']].apply(
    lambda x: True if (x['lon'], x['lat']) in bau_substations_loc.tolist() else False, 
    axis=1
)
substation_df = asset_gm_df[substation_mask]
# Add "type" as "Substation" # one substation per bus
substation_df['type'] = "Substation"
# Add "Voltage" as v_nom
substation_df['Voltage'] = substation_df['v_nom_kv']
# Set "Anchored" as true
substation_df['Anchored'] = "true"
number_of_substations = len(substation_df)

# generation
generation_df = asset_gm_df[asset_gm_df['asset_type'] == 'generator']
generation_df = generation_df[generation_df['new_mw'] > 0]
generation_df['id'] = generation_df['asset_id'].apply(
    lambda x: f"{x+number_of_substations}"
)
id_col = generation_df.pop('id')
generation_df.insert(0, 'id', id_col)
number_of_generators = len(generation_df)

# Add "type" as "Generation"
generation_df['type'] = "Generation"
# Add "Output" as v_nom
generation_df['Output'] = generation_df['new_mw'].apply(lambda x: f"{x*1e6} w")
# Set "Anchored" as true
generation_df['Anchored'] = "true"

# storage unit
storage_df = asset_gm_df[asset_gm_df['asset_type'] == 'storage_unit']
storage_df['id'] = storage_df['asset_id'].apply(
    lambda x: f"{x+number_of_substations+number_of_generators}"
)
id_col = storage_df.pop('id')
storage_df.insert(0, 'id', id_col)

# Add "type" as "Generation" (Use the same fragility as small generation plants)
storage_df['type'] = "Generation"
# Add "Capacity" as v_nom
storage_df['Output'] = f"{50*1e6} w"
# Set "Anchored" as true
storage_df['Anchored'] = "true"

asset_gm_df = pd.concat([substation_df, generation_df, storage_df], ignore_index=True)

display(asset_gm_df.head(1))
print(number_of_substations, number_of_generators, len(storage_df))

,id,scenario,network_kind,plant_id,asset_type,bus,lon,lat,v_nom_kv,region,...,marginal_cost_storage,state_of_charge_initial,state_of_charge_set,standing_loss,inflow,site_pga,asset_id,Voltage,Anchored,Output
0,0,aggressive,disagg,p1 CCGT,generator,p1,-121.532628,47.793525,230.0,wecc_exterior,...,NaN,NaN,NaN,NaN,NaN,0.0,0,230.0,true,NaN


2511 712 270


In [157]:
asset_info_csv_dir = os.path.join(eq_scenario_dir, expansion_scenario)
asset_info_csv_fn = "asset_info_pelicun.csv"
asset_gm_df.to_csv(os.path.join(asset_info_csv_dir, 
    asset_info_csv_fn),
                      index=False)

In [158]:
# Modify pypsa_config related to inventory files as needed
pypsa_config['Applications']['Assets']['Buildings'][
        'ApplicationData'
    ]['pathToSource'] = asset_info_csv_dir
pypsa_config['Applications']['Assets']['Buildings'][
    'ApplicationData'
    ]['assetSourceFile'] = asset_info_csv_fn
pypsa_config['Applications']['Assets']['Buildings'][
            'ApplicationData'
        ].pop('filter', None)
# Modify pypsa_config dl method
pypsa_config['Applications']['DL']['Buildings']['ApplicationData'][
        'DL_Method'
    ] = 'Hazus Earthquake - Electric Power'
# Modify pypsa_config variables to only save Damage
pypsa_config['SaveVariables'] = ['Demand','Damage']
# Modify number of damage state samples
pypsa_config['Applications']['DL']['Buildings'][
        'ApplicationData'
    ]['Realizations'] = num_samples


In [159]:
# Create the gm sample file
pga_samples = pd.DataFrame(
    # Manually make it two identical realization to avoid pelicun error
    # np.vstack([asset_gm_df['site_pga'].values.reshape(1, -1)]*2),
    asset_gm_df['site_pga'].values.reshape(1, -1),
    columns=asset_gm_df['id'].values
)
new_columns = pd.MultiIndex.from_product([pga_samples.columns, ['PGA']])
pga_samples.columns = new_columns
display(pga_samples.head(3))

pga_samples.to_csv(
    os.path.join(eq_scenario_dir, expansion_scenario, "pga_samples_pelicun.csv"),
    index=False,
)
# Replace the RegionalMapping application name to UserDefinedDemandSample
pypsa_config['Applications']['RegionalMapping']['Buildings'][
        'Application'] = 'UserDefinedDemandSample'
pypsa_config['RegionalEvent']['eventFilePath'] = asset_info_csv_dir
pypsa_config['RegionalEvent']['eventFile'] = f'pga_samples_pelicun.csv'

pypsa_config_fp = os.path.join(
    asset_info_csv_dir, f"pypsa_config_pelicun.json"
)
with open(pypsa_config_fp, 'w') as f:
        json.dump(pypsa_config, f, indent=4)

,0,1,2,3,4,5,6,7,8,9,...,6503,6504,6505,6506,6507,6508,6509,6510,6511,6512
,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,...,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [160]:
pga_samples.max().max()

np.float64(0.6524212646484375)

In [161]:
# command to run pelicun
pelicun_path = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/python_libs/pelicun_cli.py"
python_path = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python"

run_dir = asset_info_csv_dir
curdir = os.getcwd()
os.chdir(run_dir)
command = f"{python_path} -u {pelicun_path} regional_sim {pypsa_config_fp} -n 1"
try:
    subprocess.run(command, shell=True, check=True)
except subprocess.CalledProcessError as e:
    print(f"Command failed with error: {e}")

os.chdir(curdir)

/resnick/groups/enceladus/jyzhao/CAPowerResilience/python_libs/pelicun/pelicun/__init__.py:61: PelicunWarning: DLML data update available. Current version: v2.1.0, Latest version: v3.2. Update with: /resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python -m pelicun.cli dlml update
  check_dlml_data()


DLML data update available. Current version: v2.1.0, Latest version: v3.2. Update with: /resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python -m pelicun.cli dlml update
[00:00:00] 2 Building Inventory


Processing building chunks: 100%|██████████| 4/4 [01:36<00:00, 24.16s/it]


[00:01:37] 7 Save results


In [162]:
asset_gm_df['id'] = asset_gm_df['id'].astype(int)
damage_id_to_asset_id = asset_gm_df.set_index('id')['asset_id'].to_dict()

In [163]:
damage_df_all = pd.read_csv(os.path.join(
    asset_info_csv_dir, "damage.csv"
))

substation_damage_df = damage_df_all[damage_df_all['id'].isin(
    asset_gm_df.iloc[:number_of_substations]['id']
)].copy()
substation_damage_df['asset_id'] = substation_damage_df['id'].map(damage_id_to_asset_id)
generator_damage_df = damage_df_all[damage_df_all['id'].isin(
    asset_gm_df.iloc[number_of_substations:number_of_substations+number_of_generators]['id']
)].copy()
generator_damage_df['asset_id'] = generator_damage_df['id'].map(damage_id_to_asset_id)
storage_damage_df = damage_df_all[damage_df_all['id'].isin(
    asset_gm_df.iloc[number_of_substations+number_of_generators:]['id']
)].copy()
storage_damage_df['asset_id'] = storage_damage_df['id'].map(damage_id_to_asset_id)
print(substation_damage_df.shape, generator_damage_df.shape, storage_damage_df.shape)

substation_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_substation.csv"), index=False)
generator_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_generator.csv"), index=False)
storage_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_storage.csv"), index=False)

(2511, 1002) (712, 1002) (270, 1002)


In [164]:
# Get remaining functional capacity according to Table 8-28 in HAZUS 6.1
substation_damage_to_remaining_functional_capacity = {
    0: 1.0,
    1: 0.5,
    2: 0.09,
    3: 0.04,
    4: 0.03
}
generator_damage_to_remaining_functional_capacity = {
    0: 1.0,
    1: 1.0,
    2: 0.24,
    3: 0.16,
    4: 0.02
}
storage_damage_to_remaining_functional_capacity = generator_damage_to_remaining_functional_capacity

for i in tqdm(range(num_samples), total=num_samples):
    col_name = str(i)
    substation_damage_df[col_name] = substation_damage_df[col_name].map(
        substation_damage_to_remaining_functional_capacity)
    generator_damage_df[col_name] = generator_damage_df[col_name].map(
        generator_damage_to_remaining_functional_capacity)
    storage_damage_df[col_name] = storage_damage_df[col_name].map(
        storage_damage_to_remaining_functional_capacity)

100%|██████████| 1000/1000 [00:03<00:00, 329.53it/s]


In [165]:
substation_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_substation.csv"), index=False)
generator_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_generator.csv"), index=False)
storage_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_storage.csv"), index=False)